# Leo 2 — русский + английский (обучение с нуля, модель base ~41 млн параметров)

Запускайте ячейки **по порядку**. Нужен GPU (T4): Среда выполнения → Сменить тип среды.
Прогресс сохраняется в ваш **приватный** репозиторий Hugging Face, поэтому сброс Colab не страшен:
просто начните с ячейки 1 и дойдите до того места, где остановились (готовые шаги пропускаются).

Честно: это маленькая модель. Она научится связно писать по-русски и по-английски и отвечать на простые вопросы,
но «знать всё» не будет и иногда будет ошибаться.


In [ ]:
# 1. Проверка GPU, код, зависимости, токен Hugging Face (Write: huggingface.co/settings/tokens)
!nvidia-smi -L
%cd /content
!rm -rf /content/leo.ai && git clone -q https://github.com/harringtonov-cloud/leo.ai
%cd /content/leo.ai/leo-model
!pip install -q tokenizers datasets huggingface_hub
import getpass, os
os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face token (Write): ")
from huggingface_hub import HfApi
USER = HfApi(token=os.environ["HF_TOKEN"]).whoami()["name"]
TRAIN_REPO = f"{USER}/leo-train"   # приватный, для сохранения прогресса
PUBLIC_REPO = f"{USER}/leo"        # публичный, откуда сайт берёт готовую модель
print("OK:", USER)

In [ ]:
# 2. Восстановить прогресс (если он уже был). Сообщения "not found" на первом запуске - это нормально.
!python hf_sync.py pull --repo {TRAIN_REPO}

In [ ]:
# 3. Данные: русский + английский, токенизатор на 32 тысячи слов. Пропускается, если данные уже есть.
import os
if not os.path.exists("data/train.bin"):
    !python prepare.py --vocab-size 32000 --tokenizer-docs 60000 --out data \
      --source "HuggingFaceFW/fineweb-2|rus_Cyrl|text|150000|train" \
      --source "wikimedia/wikipedia|20231101.ru|text|60000|train" \
      --source "HuggingFaceFW/fineweb-edu|sample-10BT|text|100000|train" \
      --source "wikimedia/wikipedia|20231101.en|text|40000|train"
    !python hf_sync.py push --repo {TRAIN_REPO} --file data/tokenizer.json --file data/train.bin --file data/val.bin
else:
    print("Данные уже есть")

In [ ]:
# 4. Обучение. Запускайте эту ячейку снова и снова: она продолжает с места остановки.
# За один запуск идёт до 55 минут, затем прогресс сохраняется. Следите, чтобы val loss падал.
!timeout 3300 python train.py --preset base --batch-size 16 --grad-accum 4 --max-steps 6000 --eval-interval 250 --resume
!python hf_sync.py push --repo {TRAIN_REPO} --file out/ckpt.pt

In [ ]:
# 5. Попробовать, как модель продолжает текст (пока без обучения диалогу)
import torch
from leo.model import Config, GPT
from leo.tokenizer import LeoTokenizer
from leo.generate import generate_stream

def load(path):
    ck = torch.load(path, map_location="cpu")
    m = GPT(Config(**ck["config"])); m.load_state_dict(ck["model"])
    return m.to("cuda" if torch.cuda.is_available() else "cpu").eval()

tok = LeoTokenizer("data/tokenizer.json")
model = load("out/ckpt.pt"); dev = next(model.parameters()).device
for prompt in ["Москва - столица", "The capital of France is"]:
    ids = list(generate_stream(model, tok.encode(prompt), 60, 0.7, 40, 0.95, {tok.ids["<|eos|>"]}, dev))
    print(prompt + tok.decode(ids), "\n---")

In [ ]:
# 6. Обучение диалогу: русские и английские беседы (набор IlyaGusev/saiga_scored, проверьте его лицензию).
!python prepare_sft.py --out sft.jsonl --min-score 8 --limit 20000
!python sft.py --base out/ckpt.pt --tokenizer data/tokenizer.json --data sft.jsonl --epochs 2 --batch-size 8 --lr 5e-5
!python hf_sync.py push --repo {TRAIN_REPO} --file sft_out/ckpt.pt

In [ ]:
# 7. Спросить Leo по-русски
model = load("sft_out/ckpt.pt"); dev = next(model.parameters()).device
for q in ["Привет! Кто ты?", "Объясни, что такое фотосинтез, простыми словами.", "Какая столица у Франции?"]:
    ids = tok.encode_chat([{"role": "user", "content": q}], True)
    out = list(generate_stream(model, ids, 150, 0.4, 20, 0.9, {tok.ids["<|end|>"], tok.ids["<|eos|>"]}, dev))
    print("Q:", q, "\nA:", tok.decode(out), "\n---")

In [ ]:
# 8. Выложить готовую модель для сайта (публичный репозиторий: веса будут видны всем)
!python export_web.py --ckpt sft_out/ckpt.pt --out /content/leo.bin
api = HfApi(token=os.environ["HF_TOKEN"])
api.create_repo(PUBLIC_REPO, repo_type="model", exist_ok=True)
api.upload_file(path_or_fileobj="/content/leo.bin", path_in_repo="leo.bin", repo_id=PUBLIC_REPO)
api.upload_file(path_or_fileobj="data/tokenizer.json", path_in_repo="tokenizer.json", repo_id=PUBLIC_REPO)
print("Готово. Адреса те же, что у вас уже в Vercel (если репозиторий называется так же).")
print("LEO_WEIGHTS_URL =", f"https://huggingface.co/{PUBLIC_REPO}/resolve/main/leo.bin")